In [2]:
import scanpy as sc
import spapros as sp
sc.settings.verbosity = 0
sc.logging.print_header()
print(f"spapros=={sp.__version__}")

spapros==0.1.6


In [3]:
import anndata as ad

adata = ad.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad", backed='r')
# Leggi solo una vista in RAM
adata = adata[:100000, :].to_memory()

print(adata)

AnnData object with n_obs × n_vars = 100000 × 32285
    obs: 'abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y'


In [4]:
adata.obs['class'].value_counts()

class
29 CB Glut       47175
28 CB GABA       45364
27 MY GABA        3594
30 Astro-Epen     1597
31 OPC-Oligo      1279
33 Vascular        284
24 MY Glut         260
NaN                120
34 Immune          115
19 MB Glut          97
20 MB GABA          85
23 P Glut           26
26 P GABA            4
Name: count, dtype: int64

In [5]:
adata.obs['subclass'].value_counts()

subclass
314 CB Granule Glut               46923
311 CBX MLI Megf11 Gaba           30784
312 CBX MLI Cdh22 Gaba            10092
309 CB PLI Gly-Gaba                4213
295 CBN Dmbx1 Gaba                 3574
327 Oligo NN                        816
316 Bergmann NN                     777
317 Astro-CB NN                     643
326 OPC NN                          463
250 CBN Neurod2 Pvalb Glut          259
315 DCO UBC Glut                    252
310 CBX Golgi Gly-Gaba              242
NaN                                 120
333 Endo NN                         110
318 Astro-NT NN                      98
181 IC Tfap2d Maf Glut               91
334 Microglia NN                     73
330 VLMC NN                          69
331 Peri NN                          66
198 IC Six3 En2 Gaba                 60
325 CHOR NN                          56
332 SMC NN                           32
313 CBX Purkinje Gaba                31
335 BAM NN                           27
323 Ependymal NN               

In [6]:
adata.obs['supertype'].value_counts()

supertype
1155 CB Granule Glut_2              46881
1149 CBX MLI Megf11 Gaba_1          30417
1151 CBX MLI Cdh22 Gaba_1           10092
1115 CBN Dmbx1 Gaba_1                3574
1144 CB PLI Gly-Gaba_1               3544
                                    ...  
1113 MV Pax6 Gly-Gaba_1                 1
1170 Astroependymal NN_2                1
1141 DCO Il22 Gly-Gaba_1                1
1143 DCO Il22 Gly-Gaba_3                1
0693 PAG-SC Neurod2 Meis2 Glut_4        1
Name: count, Length: 83, dtype: int64

In [7]:
adata.obs['cluster'].value_counts()


cluster
5201 CB Granule Glut_2              46704
5188 CBX MLI Megf11 Gaba_1          30315
5192 CBX MLI Cdh22 Gaba_1           10092
5000 CBN Dmbx1 Gaba_1                3450
5178 CB PLI Gly-Gaba_1               2980
                                    ...  
4769 LDT-PCG St18 Gaba_1                1
4667 PB Sst Gly-Gaba_1                  1
4496 MV-SPIV Zic4 Neurod2 Glut_1        1
4148 PSV Pvalb Lhx2 Glut_1              1
2791 PAG-SC Neurod2 Meis2 Glut_4        1
Name: count, Length: 177, dtype: int64

In [8]:
adata.obs['cluster_alias'].value_counts()

cluster_alias
5158.0    46704
4167.0    30315
4170.0    10092
2173.0     3450
4164.0     2980
          ...  
4047.0        1
4026.0        1
4021.0        1
3694.0        1
5272.0        1
Name: count, Length: 177, dtype: int64

In [9]:
adata.obs['celltype'] = adata.obs['class']

# Normalize (CPM-like)
sc.pp.normalize_total(adata)

# Log-transform
sc.pp.log1p(adata)

# Select HVGs (for PCA part of Spapros)
sc.pp.highly_variable_genes(
    adata,
    flavor="cell_ranger",
    n_top_genes=1000   # recommended
)

In [10]:
adata

AnnData object with n_obs × n_vars = 100000 × 32285
    obs: 'abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y', 'celltype'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

In [11]:
selector = sp.se.ProbesetSelector(
    adata,
    n=None,               # select as many as needed
    n_pca_genes=0,        # NO PCA, only DE + tree markers
    celltype_key="celltype",
    verbosity=1
)
selector.select_probeset()

first_selection = selector.probeset.index[selector.probeset.selection].tolist()


Output()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:1482: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = a.obs.groupby(ct_key)["test_set"].value_counts()


Note: The following celltypes' test set sizes for forest training are below min_test_n (=20):
	 20 MB GABA : 17
	 23 P Glut  : 6
	 26 P GABA  : 4
The genes selected for those cell types potentially don't generalize well. Find the genes for each of those cell types in self.genes_of_primary_trees after running self.select_probeset().


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. No tree is calculated for celltype 26 P GABA.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

...

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

In [14]:
selector.probeset

,gene_nr,selection,rank,marker_rank,tree_rank,importance_score,pca_score,pre_selected,prior_selected,pca_selected,celltypes_DE_1vsall,celltypes_DE_specific,celltypes_DE,celltypes_marker,list_only_ct_marker,required_marker,required_list_marker
ENSMUSG00000028581,1,True,1.0,1.0,1.0,0.993837,0.0,False,False,False,34 Immune,,34 Immune,34 Immune,False,True,False
ENSMUSG00000046160,2,True,1.0,1.0,1.0,0.801383,0.0,False,False,False,31 OPC-Oligo,,31 OPC-Oligo,31 OPC-Oligo,False,True,False
ENSMUSG00000024998,3,True,1.0,1.0,1.0,0.539700,0.0,False,False,False,,30 Astro-Epen,30 Astro-Epen,30 Astro-Epen,False,True,False
ENSMUSG00000025255,4,True,1.0,1.0,1.0,0.535925,0.0,False,False,False,,19 MB Glut,19 MB Glut,19 MB Glut,False,True,False
ENSMUSG00000074749,5,True,1.0,1.0,1.0,0.526243,0.0,False,False,False,,20 MB GABA,20 MB GABA,20 MB GABA,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000041552,996,False,NaN,NaN,NaN,NaN,0.0,False,False,False,,,,,False,False,False
ENSMUSG00000031293,997,False,NaN,NaN,NaN,NaN,0.0,False,False,False,,,,,False,False,False
ENSMUSG00000015405,998,False,NaN,NaN,NaN,NaN,0.0,False,False,False,,,,,False,False,False
ENSMUSG00000044583,999,False,NaN,NaN,NaN,NaN,0.0,False,False,False,,,,,False,False,False


In [19]:
import numpy as np
print(np.unique(first_selection))

['ENSMUSG00000000567' 'ENSMUSG00000002985' 'ENSMUSG00000004633'
 'ENSMUSG00000007097' 'ENSMUSG00000008734' 'ENSMUSG00000016087'
 'ENSMUSG00000018398' 'ENSMUSG00000018593' 'ENSMUSG00000020354'
 'ENSMUSG00000020428' 'ENSMUSG00000020524' 'ENSMUSG00000021127'
 'ENSMUSG00000021848' 'ENSMUSG00000022112' 'ENSMUSG00000022122'
 'ENSMUSG00000024985' 'ENSMUSG00000024998' 'ENSMUSG00000025255'
 'ENSMUSG00000026473' 'ENSMUSG00000026872' 'ENSMUSG00000027200'
 'ENSMUSG00000028033' 'ENSMUSG00000028581' 'ENSMUSG00000029999'
 'ENSMUSG00000031586' 'ENSMUSG00000031765' 'ENSMUSG00000032440'
 'ENSMUSG00000032452' 'ENSMUSG00000033006' 'ENSMUSG00000035513'
 'ENSMUSG00000036469' 'ENSMUSG00000039114' 'ENSMUSG00000039488'
 'ENSMUSG00000039706' 'ENSMUSG00000041046' 'ENSMUSG00000041078'
 'ENSMUSG00000042453' 'ENSMUSG00000045092' 'ENSMUSG00000046160'
 'ENSMUSG00000046844' 'ENSMUSG00000047161' 'ENSMUSG00000052581'
 'ENSMUSG00000053025' 'ENSMUSG00000053930' 'ENSMUSG00000056966'
 'ENSMUSG00000059146' 'ENSMUSG0000006343

In [13]:
remaining = [g for g in adata.var_names if g not in first_selection]

selector2 = sp.se.ProbesetSelector(
    adata[:, remaining],
    n=150,
    n_pca_genes=100,
    celltype_key="celltype",
    verbosity=1
)
selector2.select_probeset()

second_selection = selector2.probeset.index[selector2.probeset.selection].tolist()


Output()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:1482: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = a.obs.groupby(ct_key)["test_set"].value_counts()


Note: The following celltypes' test set sizes for forest training are below min_test_n (=20):
	 20 MB GABA : 17
	 23 P Glut  : 6
	 26 P GABA  : 4
The genes selected for those cell types potentially don't generalize well. Find the genes for each of those cell types in self.genes_of_primary_trees after running self.select_probeset().


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. No tree is calculated for celltype 26 P GABA.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

...

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

In [16]:
selector2.probeset

,gene_nr,selection,rank,marker_rank,tree_rank,importance_score,pca_score,pre_selected,prior_selected,pca_selected,celltypes_DE_1vsall,celltypes_DE_specific,celltypes_DE,celltypes_marker,list_only_ct_marker,required_marker,required_list_marker
ENSMUSG00000018654,1,True,1.0,1.0,1.0,0.903730,0.068207,False,False,False,34 Immune,,34 Immune,34 Immune,False,True,False
ENSMUSG00000013523,2,True,1.0,1.0,1.0,0.736596,0.468262,False,False,False,31 OPC-Oligo,,31 OPC-Oligo,31 OPC-Oligo,False,True,False
ENSMUSG00000029212,3,True,1.0,1.0,1.0,0.708022,1.480190,False,False,True,,30 Astro-Epen,30 Astro-Epen,30 Astro-Epen,False,True,False
ENSMUSG00000070436,4,True,1.0,1.0,1.0,0.619480,0.257033,False,False,False,33 Vascular,,33 Vascular,33 Vascular,False,True,False
ENSMUSG00000026463,5,True,1.0,1.0,1.0,0.447007,0.892317,False,False,True,,27 MY GABA,27 MY GABA,27 MY GABA,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000015437,942,False,NaN,NaN,NaN,NaN,0.000424,False,False,False,,,,,False,False,False
ENSMUSG00000114772,943,False,NaN,NaN,NaN,NaN,0.000424,False,False,False,,,,,False,False,False
ENSMUSG00000076870,944,False,NaN,NaN,NaN,NaN,0.000412,False,False,False,,,,,False,False,False
ENSMUSG00000111862,945,False,NaN,NaN,NaN,NaN,0.000411,False,False,False,,,,,False,False,False


In [20]:
print(np.unique(second_selection))

['ENSMUSG00000000766' 'ENSMUSG00000000805' 'ENSMUSG00000001741'
 'ENSMUSG00000003228' 'ENSMUSG00000003657' 'ENSMUSG00000004031'
 'ENSMUSG00000010086' 'ENSMUSG00000010175' 'ENSMUSG00000013523'
 'ENSMUSG00000018634' 'ENSMUSG00000018654' 'ENSMUSG00000020181'
 'ENSMUSG00000020334' 'ENSMUSG00000020620' 'ENSMUSG00000020644'
 'ENSMUSG00000020732' 'ENSMUSG00000020844' 'ENSMUSG00000021061'
 'ENSMUSG00000021219' 'ENSMUSG00000021250' 'ENSMUSG00000021277'
 'ENSMUSG00000021508' 'ENSMUSG00000021852' 'ENSMUSG00000021959'
 'ENSMUSG00000022037' 'ENSMUSG00000022098' 'ENSMUSG00000022231'
 'ENSMUSG00000022309' 'ENSMUSG00000022656' 'ENSMUSG00000022817'
 'ENSMUSG00000022861' 'ENSMUSG00000022883' 'ENSMUSG00000023034'
 'ENSMUSG00000023232' 'ENSMUSG00000024299' 'ENSMUSG00000025812'
 'ENSMUSG00000026141' 'ENSMUSG00000026185' 'ENSMUSG00000026347'
 'ENSMUSG00000026424' 'ENSMUSG00000026463' 'ENSMUSG00000026748'
 'ENSMUSG00000026826' 'ENSMUSG00000027168' 'ENSMUSG00000027217'
 'ENSMUSG00000027298' 'ENSMUSG0000002758

In [ ]:
already = set(first_selection + second_selection)
remaining = [g for g in adata.var_names if g not in already]

selector3 = sp.se.ProbesetSelector(
    adata[:, remaining],
    n=150,
    n_pca_genes=100,
    celltype_key="celltype",
    verbosity=1
)
selector3.select_probeset()

third_selection = selector3.probeset.index[selector3.probeset.selection].tolist()


Output()

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:1482: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = a.obs.groupby(ct_key)["test_set"].value_counts()


Note: The following celltypes' test set sizes for forest training are below min_test_n (=20):
	 20 MB GABA : 17
	 23 P Glut  : 6
	 26 P GABA  : 4
The genes selected for those cell types potentially don't generalize well. Find the genes for each of those cell types in self.genes_of_primary_trees after running self.select_probeset().


/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. No tree is calculated for celltype 26 P GABA.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

...

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

/p/scratch/cjinm16/dipippo1/envs/scConcept-1/lib/python3.12/site-packages/spapros/evaluation/evaluation.py:2209: 
UserWarning: Zero cells of celltype 26 P GABA in train or test set. Celltype 26 P GABA is not included as reference
celltype.
  new_res = single_forest_classifications(

In [ ]:
final = list(set(first_selection + second_selection + third_selection))
len(final)
print(len(np.unique(final_selection)))

In [ ]:
evaluator.evaluate_probeset(final, set_id="final_500")

In [ ]:
selector3.probeset

In [ ]:
sp.pl.masked_dotplot(adata, selector)

In [ ]:
selector.plot_gene_overlap()